# Use Ollama for Web Search

1) Install ollama on your laptop (https://ollama.com/download)
2) Create a free account (https://signin.ollama.com/)
3) Create a a free API key and copy it (https://ollama.com/settings/keys)
4) Adapt the script below to loop over a file of links.

Note: This likely needs to run locally (not on HPC) because of how Ollama works. If you don't have Python installed, the easiest way to do so is by downloading Anaconda (https://www.anaconda.com/download/success?reg=skipped)

In [1]:
pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [2]:
#imports and client

import os
import pandas as pd
from ollama import Client

#retrieve API key
API_KEY = os.getenv("OLLAMA_API_KEY")

client = Client(
    host="http://localhost:11434",
    headers={"Authorization": f"Bearer {API_KEY}"}
)

tools = {
    "web_search":client.web_search,
    "web_fetch":client.web_fetch,
}

In [3]:
#read dataset

farms = pd.read_csv("filtered_dallas_farms.csv")

In [4]:
#make api key
API_KEY = "bbde8f7d702d4546b322e4fb98235f4d.2a-C9SLJm-aB0_T8sjCsKpMD"

client = Client(
    host="http://localhost:11434",
    headers={"Authorization": f"Bearer {API_KEY}"}
)

In [5]:
#redefine tools 
tools = {
    "web_search": client.web_search,
    "web_fetch": client.web_fetch
}

In [6]:
#test
print(API_KEY[:5])
print(API_KEY is None)

bbde8
False


In [7]:
#webscraping function
def research_farm(farm_name, website):
    messages = [
    {
        "role": "user", 
        "content": f""" Research this urban farm:
        Farm name: {farm_name}
    
        Website from dataset: {website}

        Use the provided website first. Do not switch to a different farm with a similar name.
        If the provided website is a farm, set is_farm to yes. 
        If the provided website is not a farm, set is_farm to no.
        Do not count a donation/cart page as produce sales unless the page clearly mentions produce, CSA, farm stand, market, restaurant sales, or food sales.
        If a website is provided, use web_fetch on that website first. 
        Do not use web_search unless the website is missing or unusable.
        Do not count donation pages, membership pages, event pages, volunteer opportunities, educational programs, or a general shopping cart as evidence of produce sales.
        The evidence field must contain a direct quote or specific fact from the website. Do not repeat the instructions or criteria from this prompt.
        If the website does not list acres, set acres_not_listed = true.
        If the website does list acres, set acres_not_listed = false.
        If the website lists acres and is more than 5 acres, set more_5_acres = true.
        If the website lists less than 5 acres, set more_5_acres = false.
        If a website has a page for its produce, or offers a membership to a CSA membership, or sells its produce to restaurants, set sells_produce = evidence found. 
        If a website has does not have page for its produce, or does not offer a membership to a CSA membership, or does not sell its produce to restaurants, 
        set sells_produce = evidence not found.
        If there is not enough evidence, set sells_produce = no evidence found.
        Do not infer produce sales. Only answer yes if the website explicitly states that produce is sold or describes a CSA, farm stand, farmers market, 
        restaurant sales, wholesale produce, or online produce sales.
        The evidence must be a direct quote or specific fact from the website that proves produce is sold. Do not use general mission statements, quotes, 
        donation pages, membership pages, or event pages as evidence of produce sales.
        If the evidence does not explicitly mention CSA, farm stand, farmers market, wholesale, restaurant sales, online produce sales, or direct produce 
        sales, set sells_produce to false.
        The evidence must be a direct quote that explicitly proves sales, such as “CSA,” “farm stand,” “farmers market,” “buy produce,” “purchase produce,” 
        “wholesale,” or “restaurant sales.” If the evidence is 
        only a mission statement, sustainability statement, donation page, event page, or general fresh food statement, set sells_produce to false.
        Return plain JSON only. Do not use fields like "name": "answer", "parameters", or "q".

        
        Determine:
        1. Does it have a website?
        2. If not, can you find an official website or social media page?
        3. Does it sell edible produce through Community Supported Agriculture (CSA), a farm stand, farmers market, online store, restaurants, or direct sales?
        4. If it only sells flowers or honey, do not classify it as selling produce.
        5. Does it only donate produce, provide education, or run community programs?
        6. If there is no useful information, say "no evidence found."

        Return a short answer with:
    - website_found:
    - is_farm:
    - acres_not_listed:
    - more_5_acres:
    - social_media_found:
    - acres_not_listed
    - sells_produce:
    - sells_only_flowers_or_honey:
    - evidence:
    - classification:
        """
        }
    ]

    while True: 
        response = client.chat(
            model="llama3.2:3b",
            messages=messages,
            tools=[client.web_search, client.web_fetch]
        )

        messages.append(response.message)

        if not response.message.tool_calls:
            return response.message.content
        
        for call in response.message.tool_calls:
            tool_name = call.function.name
            args = call.function.arguments

            try:
                if tool_name == "web_search":
                    result = client.web_search(**args)

                elif tool_name == "web_fetch":
                    if "url" in args:
                        result = client.web_fetch(args["url"])
                    elif "q" in args:
                        result = client.web_search(args["q"])
                    else:
                        result = f"Bad web_fetch arguments: {args}"

                else:
                    result = f"Unknown tool: {tool_name}"

            except Exception as e:
                result = f"Tool error: {e}"

            messages.append({
                "role": "tool",
                "content": str(result)
    })

In [8]:
print(farms.columns.tolist())

['name', 'address', 'lat', 'lon', 'Website', 'Google Categories', 'Place ID', 'Status', 'profile_url', 'osm_id', 'fclass', 'county', 'COUNTYFP', 'house_number', 'street', 'city', 'zipcode', 'full_address', 'Unnamed: 0', 'latitude', 'longitude', 'oid', 'geoid', 'state', 'farm_id', 'farm_name', 'street_address', 'zip_code', 'latitude_1', 'date_collected', 'website', 'index_right', 'MTFCC', 'OID', 'GEOID', 'STATE', 'COUNTY', 'COUNTYNS', 'BASENAME', 'NAME', 'LSADC', 'FUNCSTAT', 'COUNTYCC', 'AREALAND', 'AREAWATER', 'OBJECTID', 'CENTLAT', 'CENTLON', 'INTPTLAT', 'INTPTLON']


In [9]:
#when one farm works

#farms["llm_result"] = farms.apply(
    #lambda row: research_farm(
        #row["Farm"],
        #row["Website"]
    #),
   # axis=1
#)

In [10]:
client.web_fetch("https://www.okofarms.org/")

WebFetchResponse(title='Oko Farms', content="Oko Farms\n\nDONATE\n\n### URBAN FARMING, EDUCATION, AND ENVIRONMENTAL STEWARDSHIP IN BROOKLYN,NY\n\nOko Farms (est. 2013) is New York City's only publicly accessible aquaponics farm, educational center, and community hub. Using aquaponics, we sustainably grow fish and plants together in a recirculating ecosystem to save water and grow more food in small, urban spaces.\n\nThe word “oko” pays homage to our founder’s Yoruba heritage. Oko is a Yoruba word which loosely translates to farm in English. A more accurate definition of the word is a province or place where agriculture is at the center of socio-economic life, daily activities, and cultural traditions.\n\n#### MISSION\n\n#### Oko Farms’ mission is to use aquaponics farming as a tool to increase food security, combat climate change and strengthen community resilience.\n\n#### WHAT WE GROW\n\nWe cultivate a wide variety of vegetables, herbs, fruits, medicinal plants and flowers that demon

In [11]:
website = farms.loc[0, "Website"]
print(website)

client.web_fetch(website)

https://www.elmwoodfarm.co/


WebFetchResponse(title='Elmwood Farm | Join Our Sustainable Community Effort', content="Elmwood Farm | Join Our Sustainable Community Effort\n\n### Cultivating healthy relationships between Land & Neighbor in Oak Cliff, Texas\n\nJoin Our Newsletter\n\nBecome a Member Today!\n\n## Help Elmwood Farm plant long-term roots in Oak Cliff.\n\n$10.00\n\n$20.00\n\n$30.00\n\n$40.00\n\nCustom Amount\n\nPlease enter an amount\n\n$\n\nOne-Time Donation Weekly Donation Monthly Donation\n\nDonate\n\n## Elmwood Farm is a one-acre urban farm where meaningful work, play, and rest all come together.\n\n## A deeper relationship with your food and your neighbors\n\n#### Urban Farming\n\nModeling a holistic approach for truly sustainable agriculture at any scale.\n\n#### Neighborhood Events\n\nDinners, concerts, workshops, and weekly playgroups foster an open community green space.\n\n#### Community Composting\n\nWith the help of our neighbors, we divert food waste while building soil fertility.\n\n# Don’t 

In [12]:
#loop with 3
for i in range(3):
    farm_name = farms.loc[i, "name"]
    website = farms.loc[i, "Website"] if pd.notna(farms.loc[i, "Website"]) else ""

    result = research_farm(farm_name, website)

    print("------")
    print(farm_name)
    print(result)

------
Elmwood Farm
{"website_found": true, "is_farm": true, "acres_not_listed": false, "more_5_acres": false, "social_media_found": true, "sells_produce": "evidence found", "sells_only_flowers_or_honey": false, "evidence": "Modeling a holistic approach for truly sustainable agriculture at any scale.", "classification": "urban farm"}
------
Joppy Momma's Farm
{"website_found": true, "is_farm": true, "acres_not_listed": false, "more_5_acres": false, "social_media_found": false, "sells_produce": "evidence found", "sells_only_flowers_or_honey": false, "evidence": "We offer fresh produce, herbs, honey, and year-round family-friendly events...", "classification": "farm"}
------
Hatcher Station Training Farm
{"website_found": true, "is_farm": true, "acres_not_listed": false, "more_5_acres": true, "social_media_found": true, "sells_produce": "evidence found", "sells_only_flowers_or_honey": false, "evidence": "Restorative Farms is committed to developing a sustainable, community-driven urban f